In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import geopandas as gpd

from wildfire_stats import config as cfg
from wildfire_stats.data import classification as dcls
from wildfire_stats.data import io as dio
from wildfire_stats.data import rasters as drast
from wildfire_stats.fitting import fits as fits
from wildfire_stats.plotting import distributions as pdist
from wildfire_stats.plotting import landcover as plc
from wildfire_stats.reporting import summaries as rsum

P = cfg.load_paths()
A = cfg.load_assumptions()

In [ ]:
# SLOW; the static rasters are saved to P.logged_output / "static_evt".
# The original author never ran this to completion ("may have bugs") and used the
# 2024 rasters alone for the rest of the notebook.
# P.evt should contain every folder downloaded from LANDFIRE (CONUS and Alaska).
paths = drast.build_static_evt_us_auto(P.evt)
print(paths["us"])

In [ ]:
# written by notebook 5 (see the TODO there: this is df_refined, not the full fire table)
mtbs_gdf = dio.restore_truncated_columns(gpd.read_file(P.logged_output / "mtbs_classified.shp"))

mtbs_gdf.head()

In [ ]:
fires_with_evt = dcls.classify_with_evt(
    fires_gdf=mtbs_gdf,
    conus_raster_path=P.evt / "LF2024_EVT_250_CONUS" / "Tif" / "LC24_EVT_250.tif",
    alaska_raster_path=P.evt / "LF2024_EVT_250_AK" / "Tif" / "LA24_EVT_250.tif",
    column_name="EVT_class",
)

In [ ]:
counts = plc.plot_category_counts(fires_with_evt, "EVT_class")

In [ ]:
counts = fires_with_evt["EVT_class"].dropna().value_counts()
print(f"Number of EVT categories with >{A['min_fires']} fires: {(counts > A['min_fires']).sum()}")

In [ ]:
attribute_key = dio.load_evt_attributes(P.evt / "LF2024_EVT.csv")

mtbs = dio.attach_evt_attributes(fires_with_evt, attribute_key, code_col="EVT_class", key_col="VALUE")

In [ ]:
# ecological systems unit
plc.plot_category_facets(mtbs, col_name="EVT_NAME")

In [ ]:
# vegetation lifeform
plc.plot_category_facets(mtbs, col_name="EVT_LF")

In [ ]:
# Vegetation physiognomy
plc.plot_category_facets(mtbs, col_name="EVT_PHYS")

In [ ]:
# Collapsed vegetation type
plc.plot_category_facets(mtbs, col_name="EVT_GP_N")

In [ ]:
# only fit EVT_NAME groups with >= min_fires fires total
overall_results_evt = fits.fit_static_by_group(mtbs, "EVT_NAME", min_n=A["min_fires"])

dio.save_pickle(overall_results_evt, P.output / "overall_results_evtname.pkl")

In [ ]:
timevary_results_evt = fits.analyze_time_varying_mle(
    mtbs,
    overall_results_evt,
    group_col="EVT_NAME",
    restrict_to_fitted=True,
    R_boot=20,  # the original EVT run used the old default R_boot=20, not timevary_R_boot
)

In [ ]:
df_both_evt, df_p1_evt, df_p2_evt = rsum.summarize_timevary_modes(timevary_results_evt)
timevary_summary = rsum.summarize_fits(df_both_evt, df_p1_evt, df_p2_evt, overall_results_evt)
timevary_summary

The CCDF / CDF / Q–Q plotting function that was defined here is now `wildfire_stats.plotting.distributions.plot_group_ccdf_cdf_qq`.

In [ ]:
fig = pdist.plot_group_ccdf_cdf_qq(
    mtbs,
    "Apacherian-Chihuahuan Semi-Desert Shrub-Steppe",
    col_name="EVT_NAME",
    which=("weibull",),
    year_col="year",
    timevary_summary=timevary_summary,
    timevary_dist="weibull",
)

In [ ]:
overall_results_evt_collapsed = fits.fit_static_by_group(mtbs, "EVT_GP_N", min_n=A["min_fires"])

dio.save_pickle(overall_results_evt_collapsed, P.output / "overall_results_evtname_collapsed.pkl")

In [ ]:
timevary_results_evt_collapsed = fits.analyze_time_varying_mle(
    mtbs,
    overall_results_evt_collapsed,
    group_col="EVT_GP_N",
    restrict_to_fitted=True,
    R_boot=20,  # the original EVT run used the old default R_boot=20, not timevary_R_boot
)

df_both_evt_collapsed, df_p1_evt_col, df_p2_evt_col = rsum.summarize_timevary_modes(timevary_results_evt_collapsed)
timevary_summary_col = rsum.summarize_fits(df_both_evt_collapsed, df_p1_evt_col, df_p2_evt_col, overall_results_evt_collapsed)
timevary_summary_col

In [ ]:
mtbs['EVT_NAME'].nunique()

In [ ]:
mtbs['EVT_GP_N'].nunique()